In [ ]:
from typing import TypedDict,Annotated,List,Literal
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage,SystemMessage
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph,END,MessagesState
from langgraph.prebuilt import create_react_agent,ToolNode
from langgraph.checkpoint.memory import MemorySaver
import random
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
from langchain.chat_models import init_chat_model
llm=init_chat_model("groq:llama-3.1-8b-instant")
llm
class SuperVisorState(MessagesState):
    """State for the supervisor multi - agent systems"""
    current_agent:str=""
    task_assignments:dict[str,List[str]]={}#track what each agent should do
    agent_outputs:dict[str,any]={}#store outputs from each agents
    workflow_stage:str ="initial"#track workflow progress
    iteration_count:int =0 
    max_iteration:int = 10
    final_output:str = ""
class SuperVisorState(MessagesState):
    """State for the multi-agent system"""
    next_agent:str =""
    research_data:str=""
    analysis:str=""
    final_report:str=""
    task_complete:bool=False
    current_task:str=""
def create_supervisor_agent():
    """creates the supervisor decision chain"""
    supervisor_prompt= ChatPromptTemplate.from_messages([
        ("system","""you are a supervisr managing a team of agentS:
         1.Researcher - Gathers information and data
         2.Analyst - Analyzes data and provide insights         
         3.Writer - Creates reports and summaries
         based on current state and conversation , decide which worker should work next
         if the task is complete,respond with 'Done'
         Curret state:
         -Has research data:{has_research}
         -Has analysis:{has_analysis}
         -Has report:{has_report}
         Respond with ONLY the agent name(researcher/analyst/writer) or 'DONE'
         """),
                 ("human","{task}")
            ])
    return supervisor_prompt| llm
def supervisor_agent(state:SuperVisorState)->dict:
    """Supervisor decides next agent using groq llm"""
    messages=state['messages']
    task=messages[-1].content if messages else "no task"
    has_research= bool(state.get("research_data",""))
    has_analysis=bool(state.get("analysis",""))
    has_report=bool(state.get("final_data",""))
  #get LLM reports
    chain=create_supervisor_agent()
    decision = chain.invoke({
    "task": task,
   "has_research":has_research,
   "has_analysis":has_analysis,
   "has_report":has_report
   })
#Parse condition
    decision_text=decision.content.strip().lower()
    print(decision_text)
# Determine next agent
    if "done" in decision_text or has_report:
        next_agent="end"
        supervisor_msg=" Supervisor : All tasks complete! Great work team"
    elif "researcher" in decision_text or not has_research:
        next_agent="researcher"
        supervisor_msg="Supervisor: let's start with research. Assigning to researcher..."
    elif "analyst" in decision_text or (has_research and not  has_analysis) :
        next_agent="analyst"
        supervisor_msg="Supervisor: research done. Time for analysis. Assigning to analyst..."   
    
    elif "writer" in decision_text or (has_analysis and not  has_report) :
        next_agent="writer"
        supervisor_msg="Supervisor: Analysis complete. Let's create the report. Assigning to Writer..."
    else :
        next_agent="end"
        supervisor_msg="Supervisor: Task seems complete."
    return {
        "messages":[AIMessage(content=supervisor_msg)],
        "next_agent":next_agent,
        "current_task":task
    }
def researcher_agent(state:SuperVisorState)->dict:
    """researcher uses Groq to gather information"""
    task=state.get("current_task","research topic")
    #create research prompt
    research_prompt=f"""Aa a researcher specialist,provide comprehensive information about:{task}
    
    Include:
    1. Key facts and background
    2. Current trends or developments
    3. Important statistics or data points
    4. Notable examples or case studies
    Be concise but thorough  
    """
    #Get report from LLM
    research_response=llm.invoke([HumanMessage(content=research_prompt)])
    research_data=research_response.content
    #create agent message
    agent_message=f"Researche: I have completed the research on '{task}'.\n\nKey findings:\n{research_data}"
    return{
        "messages":[AIMessage(content=agent_message)],
        "research_data":research_data,
        "next_agent":"supervisor",
         
     }
def analyst_agent(state:SuperVisorState)->dict:
    """Analyst uses Groq to analyze the data"""
    research_data=state.get("research_data","")
    task=state.get("current_task","")
    #create research prompt
    analysis_prompt=f"""As a data analyst. analyze this research data and provide insights:
Research data:
{research_data}
Provide:
1.Key insights and patterns
2.Strategic implications
3.Risk and opportunities
4.Recommendations
Focus in actionable insights related to:{task}  
    """
    #Get report from LLM
    analysis_response=llm.invoke([HumanMessage(content=analysis_prompt)])
    analysis=analysis_response.content
    #create agent message
    agent_message=f"Analyst: I have completed the analysis.\n\nTop Insights:\n{analysis}"
    return{ 
        "messages":[AIMessage(content=agent_message)],
        "analysis":analysis,
        "next_agent":"supervisor",
         
     }
def writer_agent(state:SuperVisorState)->dict:
    """Writer uses Groq to create final report"""
    research_data = state.get("research_data","")
    analysis=state.get("analysis","")
    task=state.get("current_task","")
    #create writing prompt
    writer_prompt=f"""Aa a professional writer,create an executive report based on:
    
Task:{task}

Research_findings:
{research_data[:1000]}
Analysis:
{analysis[:1000]}
Create a well-structured report with:
1.Executive summary
2.key Findings
3.Analysis & Insights
4.Recommendations
5.Conclusions
Keep it professional and concise    
    """
    #Get report from LLM
    report_response=llm.invoke([HumanMessage(content=writer_prompt)])
    report=report_response.content
    #create final formatted report
    final_report=f"""
    Final Report
{'='*50}
Generated:{datetime.now().strftime('%Y-%m-%d %H:%M')}
Topic:{task}
{'='*50}
{report}
{'='*50}
Report compiled by Multi-Agent aisystem powered by Groq
""" 
    return{
        "messages":[AIMessage(content=f"writer:report complete! see below for the full document")],
        "final_report":final_report,
        "next_agent":"supervisor",
        "task_complete":True       
     }
def router(state:SuperVisorState)-> Literal["supervisor","researcher","analyst","writer","__end__"]:
    """Routes to next agent  based on state"""
    next_agent=state.get("next_agent","supervisor")
    if next_agent == "end" or state.get("task_complete",False):
        return END
    if next_agent in ["supervisor","researcher","analyst","writer"]:
        return next_agent
    return "supervisor"
workflow=StateGraph(SuperVisorState)
workflow.add_node("supervisor",supervisor_agent)
workflow.add_node("researcher",researcher_agent)
workflow.add_node("analyst",analyst_agent)
workflow.add_node("writer",writer_agent)
workflow.set_entry_point('supervisor')

for node in ["supervisor","researcher","analyst","writer"]:
    workflow.add_conditional_edges(
        node,
        router,
        {
            "supervisor":"supervisor",
            "researcher":"researcher",
            "analyst":"analyst",
            "writer":"writer",
            END:END
        })
graph=workflow.compile()
    
graph


